## 1) 정밀도와 재현율
* 정밀도와 재현율은 Positive 데이터 세트의 예측 성능에 더 초점을 맞춘 평가 지표
* 정밀도(양성 예측도)
  * 예측을 positive로 한 대상 중 예측과 실제값이 positive로 일치한 데이터의 비율. 
  * TP/(FP+TP)
  * 실제 negative인 음성 데이터 예측을 positive 양성으로 잘못 판단했을 때 업무상 큰 영향이 발생할 경우
  * 예 : 스팸 메일 분류
  * 사이킷런의 precision_score() API
* 재현율(민감도(sensitivity), TPR(True Positive Rate))
  * 실제값이 positive인 대상 중 예측값과 실제값이 positive로 일치한 데이터 비율
  * TP/(FN+TP)
  * 실제 positive 양성 데이터를 negative로 잘못 판단 시 업무상 큰 영향이 발생할 경우
  * 예 : 암 판단 모델, 보험사기 등의 금융 사기 적발 모델
  * 사이킷런의 recall_score() API
* 재현율과 정밀도 모두 TP를 높이는 데 초점을 맞추나, 재현율은 FN을 낮추는 데, 정밀도는 FP를 낮추는 데 집중

In [2]:
### 2.6. 참고

from sklearn.preprocessing import LabelEncoder

# Null 처리 함수
def fillna(df):
    df['Age'].fillna(df['Age'].mean(), inplace=True)
    df['Cabin'].fillna('N', inplace=True)
    df['Embarked'].fillna('N', inplace=True)
    df['Fare'].fillna(0, inplace=True)
    return df

# 머신러닝 알고리즘에 불필요한 속성 제거
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# 레이블 인코딩 수행
def format_features(df):
    df['Cabin']=df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

In [3]:
# confusion_matrix, accuracy, precision, recall 등의 평가를 한 번에 하는 get_clf_eval() 함수 만들기
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

def get_clf_eval(y_test, pred):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    print('오차행렬 : ')
    print(confusion)
    print('정확도 : {0:.4f} , 정밀도 : {1:.4f} , 재현율 : {2:.4f}'.format(accuracy, precision, recall))
    
# 로지스틱 회귀 기반으로 타이타닉 생존자 예측하고 평가 수행
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

titanic_df = pd.read_csv(r'./kaggle/titanic/titanic_train.csv')
y_titanic_df = titanic_df['Survived']
x_titanic_df = titanic_df.drop('Survived', axis=1)
x_titanic_df = transform_features(x_titanic_df)

x_train, x_test, y_train, y_test = train_test_split(x_titanic_df, y_titanic_df, test_size=0.2, random_state=11)

lr_clf = LogisticRegression()

lr_clf.fit(x_train, y_train)
pred = lr_clf.predict(x_test)
get_clf_eval(y_test, pred)

오차행렬 : 
[[104  14]
 [ 13  48]]
정확도 : 0.8492 , 정밀도 : 0.7742 , 재현율 : 0.7869


C:\Users\lumos\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:762: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 2) 정밀도/재현율 트레이드오프
* 정밀도/재현율 트레이드오프(Trade-off)
  * 분류하려는 업무의 특성상 정밀도 or 재현율이 특별히 강조되어야 할 경우 분류의 결정 임겠값(threshold)을 조정해 수치를 높일 수 있음
  * 두 수치는 보완적 평가 지표이므로 어느 한쪽을 높이면 다른 하나의 수치는 떨어지기 쉬움
* 사이킷런의 분류 알고리즘 : 예측 데이터가 특정 레이블에 속하는지 계산하기 위해, 먼저 개별 레이블별로 결정 확률 구함 -> 예측 확률이 큰 레이블 값으로 예측함
  * 임곗값을 낮추면 재현율이 올라가고 정밀도가 떨어짐 : 분류 결정 임곗값 = positive 에측값을 결정하는 확률의 기준 -> positive 예측값이 많아지면 재현율 값이 높아짐
  * 기본은 0.5임
* 사이킷런의 predict_proba() 메서드 : 개별 데이터별로 예측 확률을 반환하는 메서드 <-> predict()와 비슷한 듯 다름
  * 학습이 완료된 사이킷런 Classifier 객체에서 호출 가능
  * 테스트 피처 데이터셋을 파라미터로 입력
  * 테스트 피처 레코드의 개별 클래스 예측 확률을 numpy array로 반환함. 예측 클래스 결과 값(x)
  * m(입력값의 레코드 수) X n(클래스 값 유형). 각 열은 개별 클래스의 예측 확률
  * 이진분류에서 첫번째 칼럼은 0 negative의 확률, 두 번째 칼럼은 1 positive의 확률 -> 첫번째 칼럼과 두 번째 칼럼을 더하면 1이 됨
* 임곗값 변화에 따른 평가지표 알아보기 : 사이킷런의 precision_recall_curve() API
  * 입력 파라미터 : y_true(실제 클래스값 개열. 배열의 크기=데이터 건수), probas_pred(positive 칼럼의 예측 확률 배열. 배열의 크기=데이터 건수)
  * 반환값 : 정밀도(임곗값별 정밀도 값을 배열로 반환), 재현율(임곗값별 재현율 값을 배열로 반환)
  * 정밀도와 재현율의 임계값에 따른 값 변화를 곡선 형태의 그래프로 시각화 가능

In [9]:
import numpy as np

pred_proba = lr_clf.predict_proba(x_test)
pred = lr_clf.predict(x_test)
print('predict_proba의 결과 shape : {0}'.format(pred_proba.shape))
print('predict_proba array에서 앞 3개 샘플로 추출 \n:', pred_proba[:3])

#예측확률 배열과 예측결과값 배열을 병합(concatenate)해 예측 확률과 결괏값을 한눈에 ㅗ학인
pred_proba_result = np.concatenate([pred_proba, pred.reshape(-1, 1)], axis=1)
print('두 개의 class 중에서 더 큰 확률을 클래스 값으로 예측 \n', pred_proba_result[:3])

predict_proba의 결과 shape : (179, 2)
predict_proba array에서 앞 3개 샘플로 추출 
: [[0.46162417 0.53837583]
 [0.87858538 0.12141462]
 [0.87723741 0.12276259]]
두 개의 class 중에서 더 큰 확률을 클래스 값으로 예측 
 [[0.46162417 0.53837583 1.        ]
 [0.87858538 0.12141462 0.        ]
 [0.87723741 0.12276259 0.        ]]


In [10]:
### Binarizer 클래스 활용해 정밀도/재현율 트레이드오프 이해

from sklearn.preprocessing import Binarizer

X = [[1, -1, 2],
    [2, 0, 0],
    [0, 1.1, 1.2]]
#X의 개별 원소들이 threshold값 이하이면 0, 초과이면 1을 반환
binarizer = Binarizer(threshold=1.1)
print(binarizer.fit_transform(X))

custom_threshold = 0.5
#predict_proba() 반환값의 두 번쨰 칼럼(positive 클래스 칼럼 하나만) 추출해 binarizer 적용
pred_proba_1 = pred_proba[:, 1].reshape(-1, 1)
binarizer1 = Binarizer(threshold=custom_threshold).fit(pred_proba_1)
custom_predict = binarizer1.transform(pred_proba_1)
get_clf_eval(y_test, custom_predict)

custom_threshold = 0.4
pred_proba_1 = pred_proba[:, 1].reshape(-1, 1)
binarizer1 = Binarizer(threshold=custom_threshold).fit(pred_proba_1)
custom_predict = binarizer1.transform(pred_proba_1)
get_clf_eval(y_test, custom_predict)
# 임곗값을 낮추면 재현율이 올라가고 정밀도가 떨어짐

[[0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]]
오차행렬 : 
[[104  14]
 [ 13  48]]
정확도 : 0.8492 , 정밀도 : 0.7742 , 재현율 : 0.7869
오차행렬 : 
[[99 19]
 [10 51]]
정확도 : 0.8380 , 정밀도 : 0.7286 , 재현율 : 0.8361


In [11]:
### 임곗값을 증가지켜가며 평가지표 조사
### get_eval_by_threshold() 함수 제작

thresholds = [0.4, 0.45, 0.5, 0.55, 0.6]

def get_eval_by_threshold(y_test, pred_proba_c1, thresholds):
    for custom_threshold in thresholds:
        binarizer = Binarizer(threshold=custom_threshold).fit(pred_proba_c1)
        custom_predict = binarizer.transform(pred_proba_c1)
        print("임곗값 : ", custom_threshold)
        get_clf_eval(y_test, custom_predict)
get_eval_by_threshold(y_test, pred_proba[:, -1].reshape(-1, 1), thresholds)

임곗값 :  0.4
오차행렬 : 
[[99 19]
 [10 51]]
정확도 : 0.8380 , 정밀도 : 0.7286 , 재현율 : 0.8361
임곗값 :  0.45
오차행렬 : 
[[103  15]
 [ 12  49]]
정확도 : 0.8492 , 정밀도 : 0.7656 , 재현율 : 0.8033
임곗값 :  0.5
오차행렬 : 
[[104  14]
 [ 13  48]]
정확도 : 0.8492 , 정밀도 : 0.7742 , 재현율 : 0.7869
임곗값 :  0.55
오차행렬 : 
[[109   9]
 [ 15  46]]
정확도 : 0.8659 , 정밀도 : 0.8364 , 재현율 : 0.7541
임곗값 :  0.6
오차행렬 : 
[[112   6]
 [ 16  45]]
정확도 : 0.8771 , 정밀도 : 0.8824 , 재현율 : 0.7377


In [12]:
### precision_recall_curve() API : 위 get_eval_by_threshold()와 유사한 API

from sklearn.metrics import precision_recall_curve

#레이블 값이 1일 때의 예측 확률을 추출
pred_proba_class1 = lr_clf.predict_proba(x_test)[:, 1]

#precision_recall_curve() 인자로 전달할 '실제값 데이터셋과 레이블값이 1일데 예측 확률'
#넘파이 어레이의 두번째 칼럼(칼럼 인덱스 1)값에 해당하는 데이터셋
precisions, recalls, thresholds = precision_recall_curve(y_test, pred_proba_class1)
print("반환된 분류 결정 임곗값 배열의 shape : ", thresholds.shape)

#반환된 임곗값 배열 row가 143건이므로 샘플로 10건만 추출하되, 임곗값을 15 step 으로 추출
thr_index = np.arange(0, thresholds.shape[0], 15)
print('샘플 추출을 위한 임곗값 배열의 index 10개 : ', thr_index)
print('샘플용 10개의 임곗값 : ', np.round(thresholds[thr_index], 2))

#15 step 단위로 추출된 임계값에 따른 정밀도와 재현율 값
print('샘플 임곗값별 정밀도 : ', np.round(precisions[thr_index], 3))
print('샘플 임곗값별 재현율 : ', np.round(recalls[thr_index], 3))

반환된 분류 결정 임곗값 배열의 shape :  (143,)
샘플 추출을 위한 임곗값 배열의 index 10개 :  [  0  15  30  45  60  75  90 105 120 135]
샘플용 10개의 임곗값 :  [0.1  0.12 0.14 0.19 0.28 0.4  0.56 0.67 0.82 0.95]
샘플 임곗값별 정밀도 :  [0.389 0.44  0.466 0.539 0.647 0.729 0.836 0.949 0.958 1.   ]
샘플 임곗값별 재현율 :  [1.    0.967 0.902 0.902 0.902 0.836 0.754 0.607 0.377 0.148]


In [1]:
### precision_recall_curve() API의 시각화

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline

def precision_recall_curve_plot(y_test, pred_proba_c1):
    #threshold ndarray와 이 threshold에 따른 정밀도, 재현율 ndarray 추출
    precisions, recalls, thresholds = precision_recall_curve(y_test, pred_proba_c1)
    #x축을 threshold값으로, y축을 정밀도, 재현율 값으로 각각 plot 수행. 정밀도는 점선
    plt.figure(figsize=(8, 6))
    threshold_boundary = thresholds.shape[0]
    plt.plot(thresholds, precisions[0:threshold_boundary], linestyle='--', label='precision')
    plt.plot(thresholds, recalls[0:threshold_boundary], label='recall')
    #threshold값 x축의 스케일을 0.1 단위로 변경
    start, end = plt.xlim()
    plt.xticks(np.round(np.arange(start, end, 0.1), 2))
    #각 축 라벨과 범례, 그리드 설정
    plt.xlabel("Threshold value")
    plt.ylabel("Precision and Recall Value")
    plt.legend()
    plt.grid()
    plt.show()

precision_recall_curve_plot(y_test, lr_clf.predict_proba(x_test)[:,-1])

NameError: name 'y_test' is not defined

## 3) 정밀도와 재현율의 맹점
* positive 예측의 임겠값 변경하면 정밀도/재현율 수치 변함 -> 업무 환경 따라 상호보완할 수 있는 수준에서 적용 필요